# Monitoring

## Assitant
Before we monitor anything, we need something to monitor. So we start
with a RAG pipeline that answers questions about our courses.

We won't build it from scratch. We already did that in the earlier
modules, and the flow is the same three steps as always.

First we search the FAQ for the questions most relevant to the user's
question. Then we build a prompt from that question plus the documents we
found. Finally we send it to the LLM, which gives us the answer. That's
the whole pipeline, and we reuse it as-is.

## Setting up

Two helper files carry that pipeline. `ingest.py` downloads the FAQ
dataset and builds a search index over it, and `rag_helper.py` has the
`RAGBase` class that does the search-prompt-answer loop.

If you don't have them, download them:

In [2]:
PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main'

!wget {PREFIX}'/01-agentic-rag/code/ingest.py'
!wget {PREFIX}'/01-agentic-rag/code/rag_helper.py'

--2026-07-13 18:11:15--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     738  --.-KB/s    in 0.03s   

2026-07-13 18:11:15 (25.6 KB/s) - ‘ingest.py.1’ saved [738/738]

--2026-07-13 18:11:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting 

Add dependencies:

```bash
uv add python-dotenv
```

We use `python-dotenv` to load the `OPENAI_API_KEY` from a `.env` file.

## Creating the assistant

Now we pull those two helpers together into one place. `assistant.py`
loads the data and builds the index, then hands both to `RAGBase`. We
don't pass our own instructions here. `RAGBase` already comes with a
system prompt telling the model to answer course questions. A second one
would be redundant.

Create `assistant.py`.

Imports:

```python
import sys

from dotenv import load_dotenv
from openai import OpenAI

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
```

A function to create the assistant:

```python
def create_assistant():
    load_dotenv()

    documents = load_faq_data()
    index = build_index(documents)

    return RAGBase(
        index=index,
        llm_client=OpenAI(),
    )
```

Test it from the command line:

```python
if __name__ == "__main__":
    assistant = create_assistant()

    query = "How do I join the course?"
    if len(sys.argv) > 1:
        query = sys.argv[1]

    answer = assistant.rag(query)
    print(answer)
```

Run the assistant:

```bash
uv run python assistant.py
```

We'll run this command again and again, and typing it in full every time
gets old. So we put it in a `Makefile`.

Add a `run` target:

```makefile
run:
	uv run python assistant.py
```

Now we can run:

```bash
make run
```

Or with a custom question:

```bash
uv run python assistant.py "How do I join the course?"
```

You should see an answer printed to the console. Running it from the
command line is fine for us, but it's not how a user would reach it.
Next we put a simple interface in front of it with Streamlit.


# Chat App
The command line works, but we want something closer to how a person
would actually talk to the assistant. So we wrap it in a small web
interface. We use Streamlit, a Python framework for building front ends
with almost no code.

This isn't the final product, and it isn't meant to be pretty. I kept it
deliberately simple so there's nothing to explain. If you want a nicer
interface, hand it to a coding assistant like Claude Code or Codex. Ask
it to improve the layout. For now, plain is fine.

Add Streamlit to your project:

In [1]:
!uv add streamlit

Resolved 193 packages in 1.59s                                       
⠙ Preparing packages... (0/19)                                                  ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/19)------------------     0 B/8.26 KiB            
⠙ Preparing packages... (0/19)--------- 8.26 KiB/8.26 KiB           
⠙ Preparing packages... (0/19)--------- 8.26 KiB/8.26 KiB           
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/19)------------------     0 B/28.25 KiB           
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/19)------------------     0 B/28.25 KiB           
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/19)------------------     0 B/28.25 KiB           
blinker              ------------------------------ 8.26 KiB/8.26 KiB
tenacity             ------------------

Run the app:


In [2]:
!uv run streamlit run app.py


      👋 Welcome to Streamlit!

      If you'd like to receive helpful onboarding emails, news, offers, promotions,
      and the occasional swag, please enter your email address below. Otherwise,
      leave this field blank.

      Email: ^C
